# Lượt 6 — ba cặp E1/E3, seed 43/44/45

Đọc `reports/ROUND_6_CASIA_RUNBOOK.md`. Chọn **A100 GPU** và chạy từng cell.
**Chỉ chạy 0.1–1.4, dừng tại PREFLIGHT PASS rồi gửi output để kiểm tra.**
Không Run all. Train và test mặc định đóng. Ngân sách 2–3 giờ là ước lượng,
chưa đo. Không trộn seed 42 vào đợt mới. CASIA đã quan sát là development evidence.
E3 giữ nguyên lựa chọn từ validation Lượt 5, kể cả khi kết quả mới âm tính.


In [1]:
#@title 0.1 Mount Drive và đường dẫn
from google.colab import drive, files
drive.mount("/content/drive")
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, zipfile
from pathlib import Path
REPO_ROOT = Path("/content/round6-source/DeepFace-PAD")
DRIVE_ROOT = Path("/content/drive/MyDrive/face-pad")
DATA_ROOT = Path("/content/datasets/casia-fasd")
RUNS_DIR = DRIVE_ROOT / "runs"
MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug.csv"
DEPTH_MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug_with_depth.csv"
DEPTH_ROOT = DATA_ROOT / "depth"
DEPTH_LEDGER = DEPTH_ROOT / "depth_status.csv"
def run(command, cwd=REPO_ROOT, env=None):
    subprocess.run([str(x) for x in command], cwd=cwd, env=env, check=True)
def checksum(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
def safe_zip(path, target):
    target = Path(target).resolve()
    with zipfile.ZipFile(path) as archive:
        for entry in archive.infolist():
            destination = (target / entry.filename).resolve()
            assert destination.is_relative_to(target), "ZIP path outside destination"
            assert (entry.external_attr >> 16) & 0o170000 != 0o120000, "ZIP symlink"
        archive.extractall(target)


Mounted at /content/drive


In [2]:
#@title 0.2 Nạp ZIP đúng commit và cài package
archive = Path("/content/DeepFace-PAD-Round6-source.zip")
drive_archive = DRIVE_ROOT / "handoffs/round6-20261022/DeepFace-PAD-Round6-source.zip"
if not archive.is_file() and drive_archive.is_file():
    shutil.copy2(drive_archive, archive)
if not archive.is_file():
    print("Upload DeepFace-PAD-Round6-source.zip được giao cùng notebook.")
    files.upload()
assert archive.is_file(), "Thiếu source ZIP Lượt 6"
if not REPO_ROOT.exists():
    safe_zip(archive, REPO_ROOT.parent)
else:
    with zipfile.ZipFile(archive) as source_zip:
        assert source_zip.read("DeepFace-PAD/SOURCE_LOCK.json") == (REPO_ROOT / "SOURCE_LOCK.json").read_bytes(), "Source cũ khác ZIP; dừng kiểm tra"
lock = json.loads((REPO_ROOT / "SOURCE_LOCK.json").read_text())
assert all(checksum(REPO_ROOT / name) == value for name, value in lock["files"].items())
SOURCE_COMMIT = (REPO_ROOT / "SOURCE_COMMIT.txt").read_text().strip()
assert SOURCE_COMMIT == lock["commit"] and len(SOURCE_COMMIT) == 40
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
importlib.invalidate_caches()
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from deepface_pad.round6 import Batch, summarize
OUTPUT_DIR = DRIVE_ROOT / "reports" / f"round6-{SOURCE_COMMIT[:12]}"
batch = Batch(REPO_ROOT, OUTPUT_DIR)
print("Source commit:", SOURCE_COMMIT, "ZIP SHA256:", checksum(archive))


Upload DeepFace-PAD-Round6-source.zip được giao cùng notebook.


Saving DeepFace-PAD-Round6-source.zip to DeepFace-PAD-Round6-source.zip
Source commit: a006149996686cfc9e6dbd29f8fd5c90bf1997ba ZIP SHA256: fdc0de363b3db2822a380790e112ba4111eebd6e9ee3615501e36f280750afdc


In [3]:
#@title 0.3 Tests CPU, chưa train; kiểm đúng A100
cpu_env = dict(os.environ, CUDA_VISIBLE_DEVICES="")
TESTS = ['tests/test_round6.py', 'tests/test_round6_notebook.py', 'tests/test_preflight.py', 'tests/test_training_stages.py', 'tests/test_official_depth_head.py', 'tests/test_metrics.py', 'tests/test_manifest.py', 'tests/test_depth_jobs.py', 'tests/test_depth_qa.py']
run([sys.executable, "-m", "pytest", "-q", *TESTS], env=cpu_env)
assert torch.cuda.is_available(), "Chọn A100 GPU, không chạy train bằng CPU"
assert "A100" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__, "CUDA:", torch.version.cuda)


GPU: NVIDIA A100-SXM4-40GB torch: 2.11.0+cu130 CUDA: 13.0


In [4]:
#@title 1.1 Khôi phục ZIP dataset cũ trên Drive
def looks_like_casia(path):
    return (path / "train/live").is_dir() and (path / "test/spoof").is_dir()
if not looks_like_casia(DATA_ROOT):
    dataset_zip = DRIVE_ROOT / "casia-fasd.zip"
    assert dataset_zip.is_file(), "Thiếu face-pad/casia-fasd.zip; dừng, không tải bộ dữ liệu khác"
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    safe_zip(dataset_zip, DATA_ROOT.parent)
    if not looks_like_casia(DATA_ROOT):
        roots = [p.parent for p in DATA_ROOT.parent.rglob("train") if looks_like_casia(p.parent)]
        assert len(roots) == 1, f"CASIA root không duy nhất: {roots}"
        shutil.copytree(roots[0], DATA_ROOT, dirs_exist_ok=True)
assert looks_like_casia(DATA_ROOT)
print("Dataset:", DATA_ROOT)


Dataset: /content/datasets/casia-fasd


In [5]:
#@title 1.2 Manifest cùng split và sampling đã khóa
run([sys.executable, "scripts/prepare_casia_fasd.py", "--data-root", DATA_ROOT,
     "--output-dir", REPO_ROOT / "data/manifests", "--frames-per-video", "20",
     "--val-subjects", "4,9,14,19"])
from deepface_pad.round6 import SOURCE_MANIFEST, DEPTH_MANIFEST as EXPECTED_DEPTH
assert checksum(MANIFEST) == SOURCE_MANIFEST, "Manifest khác; không tự đổi split"


In [6]:
#@title 1.3 Khôi phục cache cũ; không chạy worker 3DDFA
cache_tar = DRIVE_ROOT / "depth-checkpoints/casia-fasd-depth-checkpoint.tar"
assert cache_tar.is_file(), "Thiếu TAR cache đã kiểm toán; dừng"
assert checksum(cache_tar) == "26cfa7f26d61e3a71372404008f5dc9dd696af931172850986a8642e440b3d80"
local_tar = Path("/content/casia-fasd-depth-checkpoint.tar")
if not local_tar.exists():
    shutil.copy2(cache_tar, local_tar)
assert checksum(local_tar) == checksum(cache_tar)
with tarfile.open(local_tar) as archive:
    for member in archive.getmembers():
        destination = (DATA_ROOT / member.name).resolve()
        assert destination.is_relative_to(DATA_ROOT.resolve())
        assert member.isfile() or member.isdir(), "Cache entry không phải file/directory"
        if member.isdir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            payload = archive.extractfile(member).read()
            if destination.exists():
                assert hashlib.sha256(payload).hexdigest() == checksum(destination), f"Cache cục bộ khác: {destination}"
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(payload)
worker = DRIVE_ROOT / "depth-checkpoints/casia-fasd/depth_status.csv.worker.json"
if worker.is_file() and not (DEPTH_ROOT / worker.name).exists():
    shutil.copy2(worker, DEPTH_ROOT / worker.name)
# Helper tái sử dụng live maps; chỉ khôi phục attack zero cố định/ledger.
run([sys.executable, "scripts/generate_depth.py", MANIFEST,
     "--data-root", DATA_ROOT, "--output-root", DEPTH_ROOT])
ledger = pd.read_csv(DEPTH_LEDGER, keep_default_na=False)
assert ledger.status.value_counts().to_dict() == {"complete_attack_zero":9000, "complete_bona_fide":3000}, "Cache chưa đủ; không sinh lại live depth"
run([sys.executable, "scripts/materialize_depth_manifest.py", MANIFEST,
     "--ledger", DEPTH_LEDGER, "--data-root", DATA_ROOT, "--output", DEPTH_MANIFEST])
assert checksum(DEPTH_MANIFEST) == EXPECTED_DEPTH


In [7]:
#@title 1.4 PREFLIGHT: đăng ký sáu config/source/data/environment — DỪNG
registration = batch.register(DATA_ROOT, RUNS_DIR)
identity = registration["environment"]
display(pd.DataFrame([{"key":key, "seed":cfg["seed"], "model":cfg["model"]["name"],
                      "initializer":cfg.get("round6_initializer", "random, đúng seed")}
                     for key,cfg in registration["configs"].items()]))
print("PREFLIGHT PASS", json.dumps({"source_commit":SOURCE_COMMIT,
    "gpu":identity["gpu"], "driver":identity["driver_inventory"], "torch":identity["torch"],
    "cuda":identity["cuda"], "cudnn":identity["cudnn"], "seeds":registration["seeds"],
    "depth_targets":len(registration["depth"]["artifacts"]), "epoch_budget":138,
    "registration":str(batch.registration_path)}, indent=2))
print("DỪNG: gửi output cell này và CPU tests; kiểm budget A100 trước train. Không Run all.")


,key,seed,model,initializer
0,E1_43,43,cdcn_official,"random, đúng seed"
1,E3_43,43,cdcn_official_head,E1_43
2,E1_44,44,cdcn_official,"random, đúng seed"
3,E3_44,44,cdcn_official_head,E1_44
4,E1_45,45,cdcn_official,"random, đúng seed"
5,E3_45,45,cdcn_official_head,E1_45


PREFLIGHT PASS {
  "source_commit": "a006149996686cfc9e6dbd29f8fd5c90bf1997ba",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "driver": "GPU-6a1c0d2b-a111-81de-3aa9-518baab10e43, NVIDIA A100-SXM4-40GB, 580.82.07, 40960 MiB",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "cudnn": 92700,
  "seeds": [
    43,
    44,
    45
  ],
  "depth_targets": 12000,
  "epoch_budget": 138,
  "registration": "/content/drive/MyDrive/face-pad/reports/round6-a00614999668/registration.json"
}
DỪNG: gửi output cell này và CPU tests; kiểm budget A100 trước train. Không Run all.


## 2. Train từng cặp — chỉ mở sau khi preflight đã được kiểm tra

Mỗi cặp E1 → E3 phải dùng cùng seed. Giữ partial log khi bị ngắt; không full resume.
Sau mỗi cặp, kiểm output và thời gian còn lại trước seed tiếp theo.


In [8]:
#@title 2.1 Cặp seed 43 — cổng mặc định đóng
RUN_SEED43 = True
pair43 = batch.train_pair(43, allow_train=RUN_SEED43)
for key, folder in pair43.items():
    print(key, folder)
    display(pd.read_csv(folder / "train_log.csv"))
print("DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.")


[joint] epoch 1/30 train_loss=0.077174 val_loss=0.072966 best=0.072966 elapsed=49.2s stage_eta=23.8m
[joint] epoch 2/30 train_loss=0.070849 val_loss=0.071332 best=0.071332 elapsed=47.5s stage_eta=22.2m
[joint] epoch 3/30 train_loss=0.068573 val_loss=0.065918 best=0.065918 elapsed=47.5s stage_eta=21.4m
[joint] epoch 4/30 train_loss=0.058987 val_loss=0.048288 best=0.048288 elapsed=47.6s stage_eta=20.6m
[joint] epoch 5/30 train_loss=0.044606 val_loss=0.037290 best=0.037290 elapsed=47.6s stage_eta=19.8m
[joint] epoch 6/30 train_loss=0.035340 val_loss=0.028263 best=0.028263 elapsed=47.6s stage_eta=19.0m
[joint] epoch 7/30 train_loss=0.030255 val_loss=0.035730 best=0.028263 elapsed=47.5s stage_eta=18.2m
[joint] epoch 8/30 train_loss=0.024793 val_loss=0.025133 best=0.025133 elapsed=47.5s stage_eta=17.4m
[joint] epoch 9/30 train_loss=0.020822 val_loss=0.028236 best=0.025133 elapsed=47.5s stage_eta=16.6m
[joint] epoch 10/30 train_loss=0.017323 val_loss=0.024198 best=0.024198 elapsed=47.6s stage

,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.077174,0.072966,0.072966
1,joint,2,0.070849,0.071332,0.071332
2,joint,3,0.068573,0.065918,0.065918
3,joint,4,0.058987,0.048288,0.048288
4,joint,5,0.044606,0.037290,0.037290
5,joint,6,0.035340,0.028263,0.028263
6,joint,7,0.030255,0.035730,0.035730
7,joint,8,0.024793,0.025133,0.025133
8,joint,9,0.020822,0.028236,0.028236
9,joint,10,0.017323,0.024198,0.024198


E3_43 /content/drive/MyDrive/face-pad/runs/ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed43_20261022T020926Z


,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.644460,0.587446,0.561520
1,joint,2,0.409127,0.365843,0.323623
2,joint,3,0.162806,0.160067,0.130445
3,joint,4,0.062584,0.132493,0.103376
4,joint,5,0.043927,0.088694,0.068316
5,joint,6,0.018331,0.082789,0.064570
6,joint,7,0.024771,0.114432,0.081095
7,joint,8,0.020213,0.070203,0.052747
8,joint,9,0.007652,0.087488,0.069700
9,joint,10,0.005932,0.124984,0.104657


DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.


In [9]:
#@title 2.2 Cặp seed 44 — cổng mặc định đóng
RUN_SEED44 = True
pair44 = batch.train_pair(44, allow_train=RUN_SEED44)
for key, folder in pair44.items():
    print(key, folder)
    display(pd.read_csv(folder / "train_log.csv"))
print("DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.")


[joint] epoch 1/30 train_loss=0.078215 val_loss=0.073365 best=0.073365 elapsed=47.9s stage_eta=23.1m
[joint] epoch 2/30 train_loss=0.070812 val_loss=0.071828 best=0.071828 elapsed=47.9s stage_eta=22.3m
[joint] epoch 3/30 train_loss=0.065515 val_loss=0.058899 best=0.058899 elapsed=47.8s stage_eta=21.5m
[joint] epoch 4/30 train_loss=0.052369 val_loss=0.049333 best=0.049333 elapsed=47.8s stage_eta=20.7m
[joint] epoch 5/30 train_loss=0.040295 val_loss=0.034906 best=0.034906 elapsed=47.9s stage_eta=19.9m
[joint] epoch 6/30 train_loss=0.030874 val_loss=0.029580 best=0.029580 elapsed=47.8s stage_eta=19.1m
[joint] epoch 7/30 train_loss=0.026126 val_loss=0.027389 best=0.027389 elapsed=47.8s stage_eta=18.3m
[joint] epoch 8/30 train_loss=0.023193 val_loss=0.025029 best=0.025029 elapsed=47.8s stage_eta=17.5m
[joint] epoch 9/30 train_loss=0.019678 val_loss=0.030544 best=0.025029 elapsed=47.8s stage_eta=16.7m
[joint] epoch 10/30 train_loss=0.016546 val_loss=0.028005 best=0.025029 elapsed=47.8s stage

,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.078215,0.073365,0.073365
1,joint,2,0.070812,0.071828,0.071828
2,joint,3,0.065515,0.058899,0.058899
3,joint,4,0.052369,0.049333,0.049333
4,joint,5,0.040295,0.034906,0.034906
5,joint,6,0.030874,0.029580,0.029580
6,joint,7,0.026126,0.027389,0.027389
7,joint,8,0.023193,0.025029,0.025029
8,joint,9,0.019678,0.030544,0.030544
9,joint,10,0.016546,0.028005,0.028005


E3_44 /content/drive/MyDrive/face-pad/runs/ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed44_20261022T025516Z


,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.634875,0.560442,0.528596
1,joint,2,0.378849,0.304942,0.283536
2,joint,3,0.180629,0.195036,0.166131
3,joint,4,0.070989,0.109384,0.088953
4,joint,5,0.032588,0.108445,0.086765
5,joint,6,0.016508,0.084404,0.064665
6,joint,7,0.009754,0.098910,0.078876
7,joint,8,0.008293,0.152470,0.112680
8,joint,9,0.059157,0.072512,0.050116
9,joint,10,0.010220,0.087800,0.067338


DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.


In [10]:
#@title 2.3 Cặp seed 45 — cổng mặc định đóng
RUN_SEED45 = True
pair45 = batch.train_pair(45, allow_train=RUN_SEED45)
for key, folder in pair45.items():
    print(key, folder)
    display(pd.read_csv(folder / "train_log.csv"))
print("DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.")


[joint] epoch 1/30 train_loss=0.078145 val_loss=0.073519 best=0.073519 elapsed=47.9s stage_eta=23.2m
[joint] epoch 2/30 train_loss=0.070090 val_loss=0.070468 best=0.070468 elapsed=47.9s stage_eta=22.4m
[joint] epoch 3/30 train_loss=0.062036 val_loss=0.055413 best=0.055413 elapsed=47.8s stage_eta=21.5m
[joint] epoch 4/30 train_loss=0.048346 val_loss=0.042609 best=0.042609 elapsed=47.9s stage_eta=20.7m
[joint] epoch 5/30 train_loss=0.039952 val_loss=0.038794 best=0.038794 elapsed=47.8s stage_eta=19.9m
[joint] epoch 6/30 train_loss=0.034981 val_loss=0.044806 best=0.038794 elapsed=47.8s stage_eta=19.1m
[joint] epoch 7/30 train_loss=0.030845 val_loss=0.039133 best=0.038794 elapsed=47.7s stage_eta=18.3m
[joint] epoch 8/30 train_loss=0.025329 val_loss=0.036476 best=0.036476 elapsed=47.9s stage_eta=17.6m
[joint] epoch 9/30 train_loss=0.022495 val_loss=0.029045 best=0.029045 elapsed=47.8s stage_eta=16.7m
[joint] epoch 10/30 train_loss=0.019108 val_loss=0.025057 best=0.025057 elapsed=47.8s stage

,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.078145,0.073519,0.073519
1,joint,2,0.070090,0.070468,0.070468
2,joint,3,0.062036,0.055413,0.055413
3,joint,4,0.048346,0.042609,0.042609
4,joint,5,0.039952,0.038794,0.038794
5,joint,6,0.034981,0.044806,0.044806
6,joint,7,0.030845,0.039133,0.039133
7,joint,8,0.025329,0.036476,0.036476
8,joint,9,0.022495,0.029045,0.029045
9,joint,10,0.019108,0.025057,0.025057


E3_45 /content/drive/MyDrive/face-pad/runs/ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed45_20261022T034049Z


,stage,epoch,train_loss,val_loss,val_selection_loss
0,joint,1,0.616004,0.481812,0.446580
1,joint,2,0.262728,0.231992,0.198323
2,joint,3,0.075239,0.113507,0.090254
3,joint,4,0.034519,0.104129,0.075369
4,joint,5,0.038067,0.092583,0.069706
5,joint,6,0.012231,0.056100,0.037678
6,joint,7,0.008007,0.060759,0.042399
7,joint,8,0.006964,0.064026,0.044643
8,joint,9,0.053369,0.156480,0.109911
9,joint,10,0.048252,0.093249,0.065916


DỪNG: gửi run ID/log/initializer, kiểm thời gian trước cặp tiếp theo.


In [11]:
#@title 3.1 Freeze validation đủ sáu run, chưa mở test
freeze = batch.freeze()
rows = []
for key, record in freeze["runs"].items():
    folder = Path(record["run"])
    selected = json.loads((folder / "checkpoint_selection.json").read_text())["stages"][-1]
    rows.append({"key":key, "split":"validation", "unit":"video", "run":folder.name,
                 "selected_epoch":selected["epoch"], "selection_loss":selected["selection_value"],
                 **json.loads((folder / "metrics.json").read_text())})
validation = pd.DataFrame(rows)
validation.to_csv(OUTPUT_DIR / "validation.csv", index=False)
formats = {m:"{:.4%}" for m in ("apcer","bpcer","acer","eer")}
formats.update(auc="{:.6f}", threshold="{:.8f}", selection_loss="{:.6f}")
display(validation.style.format(formats).set_caption("Validation — video; APCER/BPCER/ACER/EER %, AUC không đơn vị"))
print("DỪNG: gửi bảng validation và freeze:", batch.freeze_path)


,key,split,unit,run,selected_epoch,selection_loss,threshold,apcer,bpcer,acer,eer,auc
0,E1_43,validation,video,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed43_20261022T014520Z,29,0.016191,0.26509564,0.0000%,0.0000%,0.0000%,0.0000%,1.000000
1,E3_43,validation,video,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed43_20261022T020926Z,15,0.042883,0.39832129,0.0000%,0.0000%,0.0000%,0.0000%,1.000000
2,E1_44,validation,video,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed44_20261022T023108Z,28,0.019057,0.15203722,0.0000%,0.0000%,0.0000%,0.0000%,1.000000
3,E3_44,validation,video,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed44_20261022T025516Z,11,0.043228,0.43211308,0.0000%,0.0000%,0.0000%,0.0000%,1.000000
4,E1_45,validation,video,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed45_20261022T031641Z,23,0.018810,0.23987324,0.0000%,0.0000%,0.0000%,0.0000%,1.000000
5,E3_45,validation,video,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed45_20261022T034049Z,14,0.031939,0.38984895,0.0000%,0.0000%,0.0000%,0.0000%,1.000000


DỪNG: gửi bảng validation và freeze: /content/drive/MyDrive/face-pad/reports/round6-a00614999668/comparison_freeze.json


In [12]:
#@title 4.1 Test phát triển — chỉ sau khi kiểm freeze cả sáu run
OPEN_TEST = True
test = batch.test(allow_test=OPEN_TEST)
test.to_csv(OUTPUT_DIR / "development-test.csv", index=False)
summary, paired = summarize(test)
summary.to_csv(OUTPUT_DIR / "summary-mean-sample-sd.csv")
paired.to_csv(OUTPUT_DIR / "paired-video-acer.csv")
(OUTPUT_DIR / "paired-summary.json").write_text(json.dumps(paired.attrs, indent=2))
display(test.style.format({"apcer":"{:.4%}","bpcer":"{:.4%}","acer":"{:.4%}","eer":"{:.4%}","auc":"{:.6f}"}).set_caption("CASIA development test — video"))
display(summary.style.format({(m,s):("{:.6f}" if m=="auc" else "{:.4%}") for m in ("apcer","bpcer","acer","eer","auc") for s in ("mean","std")}).set_caption("Ba seed mới — mean và sample SD, ddof=1"))
display(paired.style.format("{:.4%}").set_caption("Video ACER; delta = E3 − E1, hiển thị theo điểm phần trăm"))
print("Paired delta mean/sample SD, điểm phần trăm:", {k:100*v for k,v in paired.attrs.items()})
print("Giữ mọi kết quả, gồm delta dương. Không chọn E4 từ test; không gộp seed 42.")
print("Lưu notebook đã chạy và gửi output:", OUTPUT_DIR)


,variant,seed,run,threshold,apcer,bpcer,acer,eer,auc
0,E1,43,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed43_20261022T014520Z,0.265096,0.0000%,13.3333%,6.6667%,2.2222%,0.993827
1,E3,43,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed43_20261022T020926Z,0.398321,0.0000%,6.6667%,3.3333%,1.1111%,0.997119
2,E1,44,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed44_20261022T023108Z,0.152037,0.3704%,4.4444%,2.4074%,2.2222%,0.995021
3,E3,44,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed44_20261022T025516Z,0.432113,0.0000%,8.8889%,4.4444%,1.1111%,0.997325
4,E1,45,ROUND6_CASIA_E1_CDCN_OFFICIAL_seed45_20261022T031641Z,0.239873,0.0000%,7.7778%,3.8889%,3.3333%,0.993128
5,E3,45,ROUND6_CASIA_E3_CDCN_OFFICIAL_JOINT_BCE_seed45_20261022T034049Z,0.389849,0.0000%,6.6667%,3.3333%,2.2222%,0.994362


variant,E1,E3,delta_E3_minus_E1
seed,,,
43,6.6667%,3.3333%,-3.3333%
44,2.4074%,4.4444%,2.0370%
45,3.8889%,3.3333%,-0.5556%


Paired delta mean/sample SD, điểm phần trăm: {'delta_mean': -0.6172839506172839, 'delta_sample_sd_ddof1': 2.685717273803345}
Giữ mọi kết quả, gồm delta dương. Không chọn E4 từ test; không gộp seed 42.
Lưu notebook đã chạy và gửi output: /content/drive/MyDrive/face-pad/reports/round6-a00614999668
